In [1]:
!ls ../data/wiki*.jsonl

../data/wikipedia_synthetic10.jsonl  ../data/wikipedia_synthetic5.jsonl
../data/wikipedia_synthetic11.jsonl  ../data/wikipedia_synthetic6.jsonl
../data/wikipedia_synthetic12.jsonl  ../data/wikipedia_synthetic7.jsonl
../data/wikipedia_synthetic1.jsonl   ../data/wikipedia_synthetic8.jsonl
../data/wikipedia_synthetic2.jsonl   ../data/wikipedia_synthetic9.jsonl
../data/wikipedia_synthetic3.jsonl   ../data/wikipedia_synthetic.jsonl
../data/wikipedia_synthetic4.jsonl


In [2]:
import json
import pandas as pd
from glob import glob

PATH = "../data/wiki*.jsonl"
files = sorted(glob(PATH))
# files = [f for f in files if "8" in f or "9" in f ]

def load_json(file):
    def get_data(raw):
        line = json.loads(raw)
        return {
            "text": line["text"],
            "labels": line.get("labels"),
            "not_labels": line.get("not_labels")
        }
    with open(file, "r") as f:
        data = [get_data(line) for line in f]
    return data

files_data = [load_json(file) for file in files]
df = pd.DataFrame([i for file_data in files_data for i in file_data])

In [3]:
df.sample(5)

,text,labels,not_labels
90555,I note how Republican affiliation functions as...,"[party identity as cultural asset, regional po...","[credential accumulation cycle, professional i..."
31525,class InfantCareFormulation(BaseSurfactant):\n...,"[application_domain_declaration, pH_tolerance_...","[thermal_processing_requirement, reactant_stoi..."
81156,Born from the copper mining boom of mid-ninete...,"[origin_stories_and_founding_purposes, economi...","[architectural_evolution_and_replacement, huma..."
16349,In a judgment rendered in absentia by the Revo...,"[jurisdictional_claim, perpetrator_identificat...","[temporal_specificity, geographic_precision, w..."
45059,The numbers tell a clear story: communities th...,"[defense_of_tradition, evidence_based_argument...","[critique_of_establishment, call_for_reform, r..."


In [4]:
def merge_group(group):
    merged_labels = set().union(*group["labels"])
    merged_not_labels = set().union(*group["not_labels"])
    merged_not_labels -= merged_labels  # remove any intersection
    return pd.Series({
        "labels": sorted(merged_labels),
        "not_labels": sorted(merged_not_labels),
    })

df = (
    df.groupby("text", sort=False)
    .apply(merge_group, include_groups=False)
    .reset_index()
)
print(f"{len(df)} unique texts after merging")
df.sample(5)


92978 unique texts after merging


,text,labels,not_labels
18094,"Formed in ancient Carboniferous limestone, Whi...","[comparative_superlative, geological_significa...","[commercial_infrastructure_mention, heritage_p..."
2926,"I would state, under oath, that I observed the...","[boundary_definition, factual_assertion, geogr...","[demographic_enumeration, procedural_complianc..."
91592,Tribano is situated approximately southwest of...,[emergency response time variability],"[administrative barriers to patient mobility, ..."
53763,A 52-year-old female presented with episodic v...,"[diagnostic_uncertainty, rare_presentation, re...","[iatrogenic_factor, pharmacological_interactio..."
58183,A dispute regarding property boundaries was do...,"[civil_disturbance, geographic_precision, inci...","[fire_emergency, romania_jurisdiction, trespas..."


In [5]:
import random

random.seed(42)

# A label is present around 3x in the data, 
# so we adjust the test ratio to get ~2% test rows after the strict split 
# that also excludes rows with test labels in "not_labels"
test_ratio = 0.02 / 3 

# All labels (positive + negative)
all_pos_labels = set(l for labs in df["labels"] for l in labs)
all_neg_labels = set(l for labs in df["not_labels"] for l in labs)

full_vocab = all_pos_labels | all_neg_labels

# Only choose test labels from positive labels
# (because they need to be ground truth in test)
candidate_test_labels = list(all_pos_labels)
random.shuffle(candidate_test_labels)

n_test_labels = int(len(all_pos_labels) * test_ratio)
test_labels = set(candidate_test_labels[:n_test_labels])

# Strict split:
# Test rows = rows containing at least one test label
test_mask = df["labels"].apply(
    lambda labs: bool(set(labs) & test_labels)
)

df_test = df[test_mask].copy()

# Train rows = rows containing NO test label anywhere
train_mask = df.apply(
    lambda row: (
        len(set(row["labels"]) & test_labels) == 0
        and len(set(row["not_labels"]) & test_labels) == 0
    ),
    axis=1
)

df_train = df[train_mask].copy()

# Reset index
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# Verify
train_seen = (
    set(l for labs in df_train["labels"] for l in labs)
    |
    set(l for labs in df_train["not_labels"] for l in labs)
)

test_seen = (
    set(l for labs in df_test["labels"] for l in labs)
    |
    set(l for labs in df_test["not_labels"] for l in labs)
)

intersection = test_labels & train_seen

print("Train rows:", len(df_train))
print("Test rows :", len(df_test))
print("Held-out test labels:", len(test_labels))
print("Intersection with train:", len(intersection))

Train rows: 89454
Test rows : 1907
Held-out test labels: 1127
Intersection with train: 0


In [6]:
import pandas as pd
from IPython.display import display

train_pos = set(lab for labs in df_train["labels"]     for lab in labs)
train_neg = set(lab for labs in df_train["not_labels"] for lab in labs)
test_pos  = set(lab for labs in df_test["labels"]      for lab in labs)
test_neg  = set(lab for labs in df_test["not_labels"]  for lab in labs)

total = len(test_pos) + len(test_neg)

# For a given target set, split by how the label was seen in train
def bucket(s):
    return {
        "seen in train as positive only":          len((s & train_pos) - train_neg),
        "seen in train as negative only":          len((s & train_neg) - train_pos),
        "seen in train as both pos and neg":       len(s & train_pos & train_neg),
        "never seen in train":                     len(s - train_pos - train_neg),
    }

tp = bucket(test_pos)   # labels used as ground-truth positives in test
tn = bucket(test_neg)   # labels used as hard negatives in test
index = list(tp.keys())

counts = pd.DataFrame({
    "used as positive in test": [tp[k] for k in index],
    "used as negative in test": [tn[k] for k in index],
    "total":                    [tp[k] + tn[k] for k in index],
}, index=index)
counts.index.name = "how the label was seen in train"

ratios = (counts / total * 100).round(1).astype(str) + "%"

print(f"Unique test positive labels: {len(test_pos)}   "
      f"Unique test negative labels: {len(test_neg)}   "
      f"Total: {total}\n")
print("── Counts ──")
display(counts)
print("── % of total test labels ──")
display(ratios)


Unique test positive labels: 5876   Unique test negative labels: 5657   Total: 11533

── Counts ──


,used as positive in test,used as negative in test,total
how the label was seen in train,,,
seen in train as positive only,604,1149,1753
seen in train as negative only,1130,788,1918
seen in train as both pos and neg,1953,2749,4702
never seen in train,2189,971,3160


── % of total test labels ──


,used as positive in test,used as negative in test,total
how the label was seen in train,,,
seen in train as positive only,5.2%,10.0%,15.2%
seen in train as negative only,9.8%,6.8%,16.6%
seen in train as both pos and neg,16.9%,23.8%,40.8%
never seen in train,19.0%,8.4%,27.4%


In [7]:
import datasets

train_ds = datasets.Dataset.from_pandas(df_train)
test_ds = datasets.Dataset.from_pandas(df_test)

dataset = datasets.DatasetDict({
    "train": train_ds,
    "test": test_ds
})

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'not_labels'],
        num_rows: 89454
    })
    test: Dataset({
        features: ['text', 'labels', 'not_labels'],
        num_rows: 1907
    })
})

In [8]:
dataset.push_to_hub("alexneakameni/ZSHOT-HARDSET-v2", commit_description="Upload of ZSHOT-HARDSET-v2 with train/test split based on held-out labels.")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-v2/commit/13977fd4deb48997695843736d0050b1ec7a7f11', commit_message='Upload dataset', commit_description='Upload of ZSHOT-HARDSET-v2 with train/test split based on held-out labels.', oid='13977fd4deb48997695843736d0050b1ec7a7f11', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-v2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alexneakameni/ZSHOT-HARDSET-v2'), pr_revision=None, pr_num=None)